# Hydropower optimisation — data contract

Both notebooks use the same input and output conventions. Set `DATA_DIR` to a directory containing `Data.csv` and `Kalam Climate Data.xlsx` (or set `CLIMATE_FILE`). A sample submission CSV is optional. Run cells in order. Install dependencies in the environment before running: `pandas numpy openpyxl scikit-learn` plus `tensorflow` for the CNN or `torch lightning pytorch-forecasting` for TFT. No notebook stores credentials or mounts a drive.

The raw measurements must contain `date_time`, `Source`, `current`, `power_factor`, `kwh`, and `v_red/v_yellow/v_blue`. Climate must contain `Date Time` and the weather columns below. `kwh` is hourly summed; other electrical fields are hourly means. Gaps in the raw measurements are retained as missing targets so they cannot silently become zero consumption. Climate is interpolated along time for the available climate interval. The supplied weather file must extend through the requested forecast period.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

DATA_DIR = Path(os.environ.get("HYDRO_DATA_DIR", "data"))
DATA_FILE = DATA_DIR / "Data.csv"
CLIMATE_FILE = DATA_DIR / "Kalam Climate Data.xlsx"
SAMPLE_FILE = DATA_DIR / "SampleSubmission.csv"  # optional; ID,kwh
OUTPUT_DIR = Path(os.environ.get("HYDRO_OUTPUT_DIR", "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
HORIZON_HOURS = 744  # 31 daily forecasts; adapt to the actual competition template
VALIDATION_HOURS = 24
CLIMATE_COLS = ["Temperature (°C)", "Dewpoint Temperature (°C)",
                "U Wind Component (m/s)", "V Wind Component (m/s)",
                "Total Precipitation (mm)", "Snowfall (mm)", "Snow Cover (%)"]
EXOG = CLIMATE_COLS + ["day_sin", "day_cos"]

def calendar_features(index):
    hour = index.hour + index.minute / 60
    return pd.DataFrame({"day_sin": np.sin(2 * np.pi * hour / 24),
                         "day_cos": np.cos(2 * np.pi * hour / 24)}, index=index)


## Load and prepare hourly data

Chunks are aggregated with sums and counts so a source/hour split across CSV chunks is combined correctly. Preserve source names as strings. An observed hour has an electrical target; missing hours stay `NaN` until each model handles them explicitly.

In [ ]:
def load_hourly_data(data_file=DATA_FILE, climate_file=CLIMATE_FILE, chunksize=500_000):
    if not data_file.exists() or not climate_file.exists():
        raise FileNotFoundError(f"Provide {data_file} and {climate_file}")
    measures = ["current", "power_factor", "kwh", "voltage"]
    parts = []
    for chunk in pd.read_csv(data_file, chunksize=chunksize, parse_dates=["date_time"]):
        needed = {"date_time", "Source", "current", "power_factor", "kwh",
                  "v_red", "v_yellow", "v_blue"}
        if missing := needed.difference(chunk.columns):
            raise ValueError(f"Missing columns in Data.csv: {sorted(missing)}")
        chunk = chunk.dropna(subset=["date_time", "Source"]).copy()
        chunk["Source"] = chunk["Source"].astype(str)
        for col in needed - {"date_time", "Source"}:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce")
        chunk["voltage"] = chunk[["v_red", "v_yellow", "v_blue"]].sum(axis=1, min_count=1)
        chunk["timestamp"] = chunk["date_time"].dt.ceil("h")  # right-closed hour
        grouped = chunk.groupby(["Source", "timestamp"], sort=False)
        block = grouped[measures].agg(["sum", "count"])
        block.columns = [f"{name}_{stat}" for name, stat in block.columns]
        parts.append(block)
    if not parts:
        raise ValueError("Data.csv contains no records")
    totals = pd.concat(parts).groupby(level=[0, 1]).sum()
    hourly = pd.DataFrame(index=totals.index)
    for name in measures:
        hourly[name] = (totals[f"{name}_sum"] if name == "kwh" else
                        totals[f"{name}_sum"] / totals[f"{name}_count"].replace(0, np.nan))
        hourly.loc[totals[f"{name}_count"].eq(0), name] = np.nan
    hourly = hourly.reset_index()
    climate = pd.read_excel(climate_file, parse_dates=["Date Time"])
    missing = set(CLIMATE_COLS).difference(climate.columns)
    if missing:
        raise ValueError(f"Missing climate columns: {sorted(missing)}")
    climate = climate.set_index("Date Time")[CLIMATE_COLS].sort_index()
    climate = climate.apply(pd.to_numeric, errors="coerce")
    climate = climate.groupby(level=0).mean()
    climate = climate.resample("h").mean().interpolate(method="time", limit_area="inside")
    climate = climate.join(calendar_features(climate.index))
    hourly = hourly.join(climate, on="timestamp", how="inner")
    hourly = hourly.sort_values(["Source", "timestamp"]).reset_index(drop=True)
    if hourly.empty:
        raise ValueError("Measurements and climate have no overlapping timestamps")
    return hourly, climate

hourly, climate = load_hourly_data()
print(f"{len(hourly):,} observed source-hours; {hourly.Source.nunique()} sources")


## Forecast dates and submission checks

The default starts immediately after the last measurement and forecasts 744 hours. If the official sample submission is available, its daily dates and source IDs determine the requested span and row order. Verify the template matches this project's `YYYY-MM-DD_Source` ID pattern before exporting.

In [ ]:
def forecast_request(hourly, sample_file=SAMPLE_FILE):
    sample = pd.read_csv(sample_file) if sample_file.exists() else None
    if sample is not None:
        if "ID" not in sample or sample["ID"].duplicated().any():
            raise ValueError("Sample submission needs unique IDs")
        dates = pd.to_datetime(sample["ID"].str.slice(0, 10), format="%Y-%m-%d", errors="raise")
        sources = sample["ID"].str.slice(11)
        if not (sample["ID"].str.slice(10, 11) == "_").all():
            raise ValueError("Expected sample IDs in YYYY-MM-DD_Source format")
        if sources.isna().any() or sources.eq("").any():
            raise ValueError("Sample submission has empty source names")
        expected = {f"{d.date()}_{s}" for d, s in zip(dates, sources)}
        if expected != set(sample["ID"]):
            raise ValueError("Sample IDs do not match the expected date/source format")
        start = dates.min().normalize()
        end = dates.max().normalize() + pd.Timedelta(hours=23)
        requested_sources = set(sources)
    else:
        start = hourly.timestamp.max() + pd.Timedelta(hours=1)
        if start.hour != 0:
            raise ValueError("Daily forecast must begin at midnight; supply a sample submission or adjust the forecast start")
        end = start + pd.Timedelta(hours=HORIZON_HOURS - 1)
        requested_sources = set(hourly.Source)
    if start <= hourly.timestamp.max():
        raise ValueError("Requested forecast overlaps observed training timestamps")
    if end > climate.index.max() or start < climate.index.min():
        raise ValueError("Climate file must cover the entire forecast interval")
    missing = requested_sources - set(hourly.Source)
    if missing:
        raise ValueError(f"Unknown sources in template: {sorted(missing)[:5]}")
    future_hours = pd.date_range(start, end, freq="h")
    if sample is None and len(future_hours) % 24:
        raise ValueError("Daily submission needs a multiple of 24 hours")
    return future_hours, requested_sources, sample

def make_submission(hourly_forecast, sample):
    daily = (hourly_forecast.assign(date=hourly_forecast.timestamp.dt.normalize())
             .groupby(["date", "Source"], as_index=False)["kwh"].sum())
    counts = hourly_forecast.assign(date=hourly_forecast.timestamp.dt.normalize()).groupby(
        ["date", "Source"]).size()
    if not counts.eq(24).all():
        raise ValueError("A forecast day does not have all 24 hourly predictions")
    daily["ID"] = daily.date.dt.strftime("%Y-%m-%d") + "_" + daily.Source
    result = daily[["ID", "kwh"]]
    if sample is not None:
        if set(result.ID) != set(sample.ID):
            raise ValueError("Forecast IDs do not match the sample submission")
        result = sample[["ID"]].merge(result, on="ID", validate="one_to_one")
    if not np.isfinite(result.kwh).all():
        raise ValueError("Forecast contains missing or infinite values")
    return result

future_hours, requested_sources, sample = forecast_request(hourly)
print(f"Forecast: {future_hours.min()} to {future_hours.max()}; {len(requested_sources)} sources")


# Model B — Temporal Fusion Transformer

TFT predicts 24 `kwh` values at a time from 168 hours of history. Weather and hour-of-day are known over the decoder; electrical readings and `kwh` are observed only in the encoder. Training reserves the last 24 hours as a time-based validation period. For longer forecasts, predicted target values are fed back for subsequent 24 hour blocks; other unknown electrical covariates are carried forward from the last observed value. This is a documented approximation for recursive TFT inference.

In [ ]:
import torch
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

torch.manual_seed(SEED)
ENCODER_HOURS, DECODER_HOURS = 168, 24
UNKNOWN = ["current", "power_factor", "voltage", "kwh"]
KNOWN = EXOG
BATCH_SIZE, EPOCHS = 64, 20

# A common end time avoids data from one source entering another's validation span.
end_time = hourly.groupby("Source").timestamp.max().min()
cutoff = end_time - pd.Timedelta(hours=VALIDATION_HOURS)
frames = []
for source, group in hourly.groupby("Source"):
    group = group.set_index("timestamp").sort_index().loc[:end_time]
    if group.empty:
        continue
    group = group.reindex(pd.date_range(group.index.min(), end_time, freq="h"))
    group["Source"] = source
    # TFT needs continuous numeric inputs; reject sources with missing training or
    # validation labels rather than inventing consumption for unobserved hours.
    if (len(group) < ENCODER_HOURS + DECODER_HOURS + VALIDATION_HOURS or
        group[UNKNOWN + KNOWN].isna().any().any()):
        continue
    frames.append(group.reset_index(names="timestamp"))
if not frames:
    raise ValueError("No complete hourly TFT series; inspect data gaps and climate coverage")
train_df = pd.concat(frames, ignore_index=True)
if requested_sources - set(train_df.Source):
    raise ValueError(f"Sources missing from TFT data: {sorted(requested_sources - set(train_df.Source))[:5]}")
origin = train_df.timestamp.min()
train_df["time_idx"] = ((train_df.timestamp - origin) / pd.Timedelta(hours=1)).astype(int)
train_df["Source"] = train_df["Source"].astype(str)
print(f"TFT sources: {train_df.Source.nunique()}; cutoff: {cutoff}")


In [ ]:
cutoff_idx = int((cutoff - origin) / pd.Timedelta(hours=1))
training = TimeSeriesDataSet(
    train_df[train_df.time_idx <= cutoff_idx],
    time_idx="time_idx", target="kwh", group_ids=["Source"],
    max_encoder_length=ENCODER_HOURS, max_prediction_length=DECODER_HOURS,
    min_encoder_length=ENCODER_HOURS, min_prediction_length=DECODER_HOURS,
    static_categoricals=["Source"], time_varying_known_reals=KNOWN,
    time_varying_unknown_reals=UNKNOWN, add_relative_time_idx=True,
    add_target_scales=True, add_encoder_length=True,
)
validation = TimeSeriesDataSet.from_dataset(
    training, train_df, min_prediction_idx=cutoff_idx + 1,
    stop_randomization=True, predict=True,
)
train_loader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
val_loader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)
model = TemporalFusionTransformer.from_dataset(
    training, learning_rate=1e-3, hidden_size=16, attention_head_size=1,
    dropout=0.1, loss=QuantileLoss(), reduce_on_plateau_patience=3,
)
checkpoint = ModelCheckpoint(dirpath=OUTPUT_DIR / "tft_checkpoints", monitor="val_loss",
                             mode="min", save_top_k=1)
trainer = Trainer(max_epochs=EPOCHS, gradient_clip_val=0.1,
                  accelerator="auto", devices=1,
                  callbacks=[checkpoint, EarlyStopping(monitor="val_loss", patience=4,
                                                      mode="min")])
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
model = TemporalFusionTransformer.load_from_checkpoint(checkpoint.best_model_path)
print("Best checkpoint:", checkpoint.best_model_path)


## Test on held-out hours

`predict(mode="prediction")` returns the median `kwh` from the quantile model in original target units. The validation series ends at the shared observed `end_time`. The seasonal baseline uses the previous day's same hour.

In [ ]:
val_pred = model.predict(val_loader, mode="prediction", return_index=True,
                         trainer_kwargs={"accelerator": "auto", "devices": 1})
val_rows = []
for prediction, (_, meta) in zip(val_pred.output.detach().cpu().numpy(),
                                 val_pred.index.iterrows()):
    source = str(meta["Source"])
    start_idx = int(meta["time_idx"])
    truth = train_df[(train_df.Source == source) &
                     train_df.time_idx.between(start_idx, start_idx + DECODER_HOURS - 1)]
    if len(truth) != DECODER_HOURS:
        raise ValueError(f"Incomplete validation for {source}")
    previous = train_df[(train_df.Source == source) &
                        train_df.time_idx.between(start_idx - 24, start_idx - 1)]
    val_rows.append((source, mean_absolute_error(truth.kwh, prediction[:len(truth)]),
                     mean_absolute_error(truth.kwh, previous.kwh)))
validation_scores = pd.DataFrame(val_rows, columns=["Source", "TFT_MAE", "seasonal_MAE"])
print(validation_scores[["TFT_MAE", "seasonal_MAE"]].mean())


## Reload and forecast

The saved best checkpoint contains model weights and dataset parameters. For each 24 hour block, the decoder has future weather, calendar covariates, and placeholders for unknown values. Pytorch Forecasting excludes unknown decoder values from the forecasting features. The next block uses the preceding forecasts as its target history. Every requested source must have a complete 168 hour history immediately before the forecast start.

In [ ]:
model = TemporalFusionTransformer.load_from_checkpoint(checkpoint.best_model_path)

def forecast_tft(model, hours=future_hours, sources=requested_sources):
    if len(hours) % DECODER_HOURS:
        raise ValueError("TFT inference requires a multiple of 24 forecast hours")
    all_rows = []
    for source in sorted(sources):
        data = train_df[train_df.Source == source].copy()
        before = hours[0] - pd.Timedelta(hours=1)
        history = data[data.timestamp <= before].tail(ENCODER_HOURS).copy()
        required = pd.date_range(end=before, periods=ENCODER_HOURS, freq="h")
        if len(history) != ENCODER_HOURS or not history.timestamp.reset_index(drop=True).equals(
                pd.Series(required, name="timestamp")):
            raise ValueError(f"Incomplete forecast history for {source}")
        for offset in range(0, len(hours), DECODER_HOURS):
            stamps = hours[offset:offset + DECODER_HOURS]
            future = pd.DataFrame({"timestamp": stamps, "Source": source})
            future["time_idx"] = ((future.timestamp - origin) / pd.Timedelta(hours=1)).astype(int)
            future[KNOWN] = climate.reindex(stamps)[KNOWN].to_numpy()
            if future[KNOWN].isna().any().any():
                raise ValueError(f"Future climate missing for {source}")
            for col in UNKNOWN:
                future[col] = float(history.iloc[-1][col])
            context = pd.concat([history.tail(ENCODER_HOURS), future], ignore_index=True)
            ds = TimeSeriesDataSet.from_dataset(training, context, predict=True,
                                                stop_randomization=True)
            pred = model.predict(ds.to_dataloader(train=False, batch_size=1,
                                                 num_workers=0), mode="prediction",
                                 trainer_kwargs={"accelerator": "auto", "devices": 1})
            forecast = pred.detach().cpu().numpy().reshape(-1)
            if len(forecast) != len(stamps):
                raise ValueError(f"Unexpected TFT prediction length for {source}")
            future["kwh"] = np.maximum(0, forecast)
            all_rows.extend((stamp, source, value) for stamp, value in
                            zip(stamps, future.kwh.to_numpy()))
            history = pd.concat([history, future], ignore_index=True).tail(ENCODER_HOURS)
    return pd.DataFrame(all_rows, columns=["timestamp", "Source", "kwh"])

hourly_forecast = forecast_tft(model)
submission = make_submission(hourly_forecast, sample)
submission.to_csv(OUTPUT_DIR / "tft_submission.csv", index=False)
print(submission.head())


### Notes on the recovered design

The earlier notebook defined an unused TensorFlow `StringLookup`, hard-coded 584 sources and CUDA, had no validation or inference, and called `.save()` on a Lightning model. This version uses source categories in `TimeSeriesDataSet`, automatic device selection and Lightning checkpoints. For a final submission, verify the actual competition template and future climate dates; retrain after validation on all observations if the holdout materially improves the result.